In [40]:
import requests
import pandas as pd
import time

tournament_id = 325
target_year = "2023"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.sofascore.com/"
}

# 1. Buscar ID da Temporada
def get_season_id(t_id, year):
    url = f"https://api.sofascore.com/api/v1/unique-tournament/{t_id}/seasons"
    r = requests.get(url, headers=headers, verify=False)
    for s in r.json().get('seasons', []):
        if s.get('year') == year: return s.get('id')
    return None

season_id = get_season_id(tournament_id, target_year)

if not season_id:
    print("Temporada não encontrada.")
else:
    all_players_data = []
    
    # 2. Loop pelas 38 Rodadas
    for round_num in range(1, 39):
        print(f"Buscando jogos da Rodada {round_num}/38...", end="\r")
        url_round = f"https://api.sofascore.com/api/v1/unique-tournament/{tournament_id}/season/{season_id}/events/round/{round_num}"
        
        response = requests.get(url_round, headers=headers, verify=False)
        if response.status_code != 200:
            continue
            
        matches = response.json().get('events', [])
        
        for match in matches:
            match_id = match['id']
            match_name = f"{match['homeTeam']['name']} x {match['awayTeam']['name']}"
            
            # 3. Buscar Estatísticas de cada Jogo
            url_lineup = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
            res_lineup = requests.get(url_lineup, headers=headers, verify=False)
            
            if res_lineup.status_code == 200:
                lineup_data = res_lineup.json()
                for side in ['home', 'away']:
                    players = lineup_data.get(side, {}).get('players', [])
                    for p in players:
                        stats = p.get('statistics', {})
                        if stats:
                            player_row = {
                                "Rodada": round_num,
                                "Partida": match_name,
                                "Jogador": p['player']['name'],
                                "Time": match[f'{side}Team']['name'],
                                **stats # Inclui todas as estatísticas automáticas
                            }
                            all_players_data.append(player_row)
            
            # Delay curto para evitar bloqueio (0.3s x 380 jogos ≈ 2 min de execução)
            time.sleep(0.3)

    # 4. Salvar tudo
    if all_players_data:
        df = pd.DataFrame(all_players_data)
        # Ordenar por rodada para ficar organizado
        df = df.sort_values(by="Rodada")
        df.to_csv("brasileirao_2023.csv", index=False, encoding='utf-8-sig')
        print(f"\n\nConcluído! Arquivo gerado com {len(all_players_data)} linhas.")
    else:
        print("\nNenhum dado encontrado.")

Buscando jogos da Rodada 38/38...

Concluído! Arquivo gerado com 17407 linhas.


In [7]:
import requests
import pandas as pd
import time

# Configuracoes
tournament_id = 325 
anos_para_baixar = ["2023", "2024", "2025"] # Lista dos anos desejados
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.sofascore.com/"
}

def get_season_id(t_id, year):
    url = f"https://api.sofascore.com/api/v1/unique-tournament/{t_id}/seasons"
    r = requests.get(url, headers=headers, verify=False)
    if r.status_code == 200:
        for s in r.json().get('seasons', []):
            if s.get('year') == year: return s.get('id')
    return None

for ano in anos_para_baixar:
    print(f"\n--- Iniciando extração de {ano} ---")
    season_id = get_season_id(tournament_id, ano)
    
    if not season_id:
        print(f"Temporada {ano} não encontrada.")
        continue
        
    all_players_data = []
    
    # Percorrendo as 38 rodadas
    for round_num in range(1, 39):
        print(f"Ano {ano} - Processando Rodada {round_num}/38", end="\r")
        url_round = f"https://api.sofascore.com/api/v1/unique-tournament/{tournament_id}/season/{season_id}/events/round/{round_num}"
        
        try:
            response = requests.get(url_round, headers=headers, verify=False)
            matches = response.json().get('events', [])
            
            for match in matches:
                match_id = match['id']
                match_name = f"{match['homeTeam']['name']} x {match['awayTeam']['name']}"
                
                # Request das escalações/stats
                res_lineup = requests.get(f"https://api.sofascore.com/api/v1/event/{match_id}/lineups", headers=headers, verify=False)
                
                if res_lineup.status_code == 200:
                    lineup_data = res_lineup.json()
                    for side in ['home', 'away']:
                        players = lineup_data.get(side, {}).get('players', [])
                        for p in players:
                            stats = p.get('statistics', {})
                            if stats:
                                # Coleta de dados com seguranca (.get)
                                player_info = p.get('player', {})
                                player_row = {
                                    "Temporada": ano,
                                    "Rodada": round_num,
                                    "Partida": match_name,
                                    "Jogador": player_info.get('name'),
                                    "Time": match[f'{side}Team']['name'],
                                    "Posicao": player_info.get('position', 'N/A'), # AQUI A POSICAO
                                    **stats 
                                }
                                all_players_data.append(player_row)
                time.sleep(0.2) # Delay para evitar bloqueio
        except Exception as e:
            print(f"\nErro na rodada {round_num}: {e}")
            continue

    # Salva um arquivo para cada ano
    if all_players_data:
        df = pd.DataFrame(all_players_data)
        filename = f"brasileirao_{ano}_completo.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nFinalizado {ano}! Arquivo {filename} gerado com {len(df)} linhas.")

print("\nProcesso completo concluído com sucesso!")


--- Iniciando extração de 2023 ---
Temporada 2023 não encontrada.

--- Iniciando extração de 2024 ---
Temporada 2024 não encontrada.

--- Iniciando extração de 2025 ---
Temporada 2025 não encontrada.

Processo completo concluído com sucesso!


In [9]:
import requests
import pandas as pd
import time
import urllib3

																  
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

				 
tournament_id = 325 
anos_para_baixar = ["2023", "2024", "2025"] 
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.sofascore.com/"
}

# Dicionário de tradução baseado nas suas imagens
mapa_tatico = {
    'ST': 'Centroavante', 'LW': 'Ponta Esquerda', 'RW': 'Ponta Direita',
    'AM': 'Meia Atacante', 'ML': 'Meia Esquerda', 'MC': 'Meia Central', 
    'MR': 'Meia Direita', 'DM': 'Volante',
    'DL': 'Lateral Esquerdo', 'DC': 'Zagueiro', 'DR': 'Lateral Direito',
    'GK': 'Goleiro'
}

# Cache para não repetir requisição de jogador que já buscamos a posição
cache_posicoes = {}

def get_detailed_position(player_id):
    if player_id in cache_posicoes:
        return cache_posicoes[player_id]
    
    try:
        url = f"https://api.sofascore.com/api/v1/player/{player_id}"
        res = requests.get(url, headers=headers, verify=False, timeout=5)
        if res.status_code == 200:
            # O SofaScore entrega a posição principal tática aqui
            pos_tativa = res.json().get('player', {}).get('position', 'N/A')
            cache_posicoes[player_id] = pos_tativa
            return pos_tativa
    except:
        pass
    return "N/A"

def get_season_id(t_id, year):
    url = f"https://api.sofascore.com/api/v1/unique-tournament/{t_id}/seasons"
		
    r = requests.get(url, headers=headers, verify=False)
    if r.status_code == 200:
        for s in r.json().get('seasons', []):
            if s.get('year') == year: return s.get('id')
		   
				   
    return None

				
for ano in anos_para_baixar:
    print(f"\n--- Extraindo {ano} com Posições Táticas ---")
    season_id = get_season_id(tournament_id, ano)
	
    if not season_id: continue
												  
				
        
    all_players_data = []
    
    for round_num in range(1, 39):
        print(f"Ano {ano} - Rodada {round_num}/38 - Jogadores no Cache: {len(cache_posicoes)}", end="\r")
        url_round = f"https://api.sofascore.com/api/v1/unique-tournament/{tournament_id}/season/{season_id}/events/round/{round_num}"
        
        try:
            response = requests.get(url_round, headers=headers, verify=False)
            matches = response.json().get('events', [])
            
            for match in matches:
									  
																						 
				
                res_lineup = requests.get(f"https://api.sofascore.com/api/v1/event/{match['id']}/lineups", headers=headers, verify=False)
                
                if res_lineup.status_code == 200:
                    lineup_data = res_lineup.json()
                    for side in ['home', 'away']:
                        players = lineup_data.get(side, {}).get('players', [])
                        for p in players:
                            stats = p.get('statistics', {})
                            if stats:
                                p_obj = p.get('player', {})
                                p_id = p_obj.get('id')
                                
                                # BUSCA A SIGLA TÁTICA (ST, LW, DC...)
                                sigla_tativa = get_detailed_position(p_id)
                                
                                player_row = {
                                    "Temporada": ano,
                                    "Rodada": round_num,
                                    "Partida": f"{match['homeTeam']['name']} x {match['awayTeam']['name']}",
                                    "Jogador": p_obj.get('name'),
                                    "Time": match[f'{side}Team']['name'],
                                    "Sigla_Tatica": sigla_tativa,
                                    "Posicao_Final": mapa_tatico.get(sigla_tativa, "Outro"),
                                    **stats 
                                }
                                all_players_data.append(player_row)
                time.sleep(0.1) 
        except Exception as e:
													   
            continue

    if all_players_data:
        df = pd.DataFrame(all_players_data)
        filename = f"brasileirao_{ano}_completo.csv"
        # Garante que as colunas de identificação fiquem no começo
        cols = ["Temporada", "Rodada", "Partida", "Jogador", "Time", "Sigla_Tatica", "Posicao_Final"]
        other_cols = [c for c in df.columns if c not in cols]
        df = df[cols + other_cols]
        
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nFinalizado {ano}! Arquivo {filename} gerado com {len(df)} linhas.")

print("\nProcesso completo concluído! Suas bases para o TCC estão prontas.")


--- Extraindo 2023 com Posições Táticas ---

--- Extraindo 2024 com Posições Táticas ---

--- Extraindo 2025 com Posições Táticas ---

Processo completo concluído! Suas bases para o TCC estão prontas.


In [12]:
import requests
import pandas as pd
import time
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Mapeamento direto dos IDs de temporada encontrados na API para o Brasileirão Série A
seasons_info = [
    {'ano': '2025', 'id': 72034},
    {'ano': '2024', 'id': 58766},
    {'ano': '2023', 'id': 48982}
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.sofascore.com/"
}

mapa_tatico = {
    'ST': 'Centroavante', 'LW': 'Ponta Esquerda', 'RW': 'Ponta Direita',
    'AM': 'Meia Atacante', 'ML': 'Meia Esquerda', 'MC': 'Meia Central', 
    'MR': 'Meia Direita', 'DM': 'Volante',
    'DL': 'Lateral Esquerdo', 'DC': 'Zagueiro', 'DR': 'Lateral Direito',
    'GK': 'Goleiro'
}

def identificar_posicao_opta(x, y):
    """Lógica de conversão de coordenadas para siglas Opta"""
    if x is None or y is None: return "N/A"
    # Eixo X: Profundidade (0-100) | Eixo Y: Largura (0-100)
    if x < 15: return "GK"
    if 15 <= x < 38:
        if 30 <= y <= 70: return "DC"
        return "DL" if y < 30 else "DR"
    if 38 <= x < 65:
        if 30 <= y <= 70: return "DM" if x < 52 else "MC"
        return "ML" if y < 30 else "MR"
    if 65 <= x < 82:
        if 30 <= y <= 70: return "AM"
        return "LW" if y < 30 else "RW"
    if x >= 82:
        if 30 <= y <= 70: return "ST"
        return "LW" if y < 30 else "RW"
    return "MC"

def get_average_positions(match_id):
    url = f"https://api.sofascore.com/api/v1/event/{match_id}/average-positions"
    try:
        r = requests.get(url, headers=headers, verify=False, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return {p.get('player', {}).get('id'): {'x': p.get('averageX'), 'y': p.get('averageY')} 
                    for side in ['home', 'away'] for p in data.get('players', {}).get(side, [])}
    except: pass
    return {}

# Loop principal por temporada
for season in seasons_info:
    ano = season['ano']
    s_id = season['id']
    print(f"\n--- Extraindo Temporada {ano} (ID Sofa: {s_id}) ---")
    
    all_players_data = []
    
    # Percorre as 38 rodadas
    for round_num in range(1, 39):
        print(f"Processando Rodada {round_num}/38", end="\r")
        url_round = f"https://api.sofascore.com/api/v1/unique-tournament/325/season/{s_id}/events/round/{round_num}"
        
        try:
            res_round = requests.get(url_round, headers=headers, verify=False, timeout=10)
            events = res_round.json().get('events', [])
            
            for match in events:
                m_id = match['id']
                # Busca coordenadas e estatísticas em paralelo
                coords = get_average_positions(m_id)
                res_lineup = requests.get(f"https://api.sofascore.com/api/v1/event/{m_id}/lineups", headers=headers, verify=False, timeout=10)
                
                if res_lineup.status_code == 200:
                    lineup_json = res_lineup.json()
                    for side in ['home', 'away']:
                        for p in lineup_json.get(side, {}).get('players', []):
                            stats = p.get('statistics', {})
                            if stats:
                                p_obj = p.get('player', {})
                                p_id = p_obj.get('id')
                                c = coords.get(p_id, {'x': None, 'y': None})
                                sigla = identificar_posicao_opta(c['x'], c['y'])
                                
                                all_players_data.append({
                                    "Temporada": ano,
                                    "Rodada": round_num,
                                    "Jogador": p_obj.get('name'),
                                    "Time": match[f'{side}Team']['name'],
                                    "Avg_X": c['x'],
                                    "Avg_Y": c['y'],
                                    "Sigla_Opta": sigla,
                                    "Posicao_Real": mapa_tatico.get(sigla, "Outro"),
                                    **stats
                                })
                time.sleep(0.1) # Evitar bloqueio
        except Exception:
            continue

    if all_players_data:
        df = pd.DataFrame(all_players_data)
        filename = f"brasileirao_{ano}_opta_final.csv"
        # Reorganizar colunas para facilitar leitura
        cols = ["Temporada", "Rodada", "Jogador", "Time", "Avg_X", "Avg_Y", "Sigla_Opta", "Posicao_Real"]
        df = df[cols + [c for c in df.columns if c not in cols]]
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSucesso: {filename} gerado com {len(df)} linhas.")

print("\nExtração finalizada com sucesso!")


--- Extraindo Temporada 2025 (ID Sofa: 72034) ---
Processando Rodada 38/38
Sucesso: brasileirao_2025_opta_final.csv gerado com 17396 linhas.

--- Extraindo Temporada 2024 (ID Sofa: 58766) ---


KeyboardInterrupt: 

In [ ]:
import requests
import pandas as pd
import time
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Mapeamento direto dos IDs de temporada encontrados na API para o Brasileirão Série A
seasons_info = [
    {'ano': '2025', 'id': 72034},
    {'ano': '2024', 'id': 58766},
    {'ano': '2023', 'id': 48982}
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.sofascore.com/"
}

mapa_tatico = {
    'ST': 'Centroavante', 'LW': 'Ponta Esquerda', 'RW': 'Ponta Direita',
    'AM': 'Meia Atacante', 'ML': 'Meia Esquerda', 'MC': 'Meia Central', 
    'MR': 'Meia Direita', 'DM': 'Volante',
    'DL': 'Lateral Esquerdo', 'DC': 'Zagueiro', 'DR': 'Lateral Direito',
    'GK': 'Goleiro'
}

def identificar_posicao_opta(x, y):
    """Lógica de conversão de coordenadas para siglas Opta"""
    if x is None or y is None: return "N/A"
    # Eixo X: Profundidade (0-100) | Eixo Y: Largura (0-100)
    if x < 15: return "GK"
    if 15 <= x < 38:
        if 30 <= y <= 70: return "DC"
        return "DL" if y < 30 else "DR"
    if 38 <= x < 65:
        if 30 <= y <= 70: return "DM" if x < 52 else "MC"
        return "ML" if y < 30 else "MR"
    if 65 <= x < 82:
        if 30 <= y <= 70: return "AM"
        return "LW" if y < 30 else "RW"
    if x >= 82:
        if 30 <= y <= 70: return "ST"
        return "LW" if y < 30 else "RW"
    return "MC"

def get_average_positions(match_id):
    url = f"https://api.sofascore.com/api/v1/event/{match_id}/average-positions"
    try:
        r = requests.get(url, headers=headers, verify=False, timeout=10)
        if r.status_code == 200:
            data = r.json()
            coords = {}
            for side in ['home', 'away']:
                players_list = data.get(side, [])
                for p in players_list:
                    # FORÇAMOS O ID PARA STRING AQUI
                    p_id = str(p.get('player', {}).get('id'))
                    if p_id:
                        coords[p_id] = {
                            'x': p.get('averageX'),
                            'y': p.get('averageY')
                        }
            return coords
    except Exception as e:
        print(f"Erro ao buscar coordenadas: {e}")
    return {}

# Loop principal por temporada
for season in seasons_info:
    ano = season['ano']
    s_id = season['id']
    print(f"\n--- Extraindo Temporada {ano} (ID Sofa: {s_id}) ---")
    
    all_players_data = []
    
    # Percorre as 38 rodadas
    for round_num in range(1, 39):
        print(f"Processando Rodada {round_num}/38", end="\r")
        url_round = f"https://api.sofascore.com/api/v1/unique-tournament/325/season/{s_id}/events/round/{round_num}"
        
        try:
            res_round = requests.get(url_round, headers=headers, verify=False, timeout=10)
            events = res_round.json().get('events', [])
            
            for match in events:
                m_id = match['id']
                coords = get_average_positions(m_id)
                res_lineup = requests.get(f"https://api.sofascore.com/api/v1/event/{m_id}/lineups", headers=headers, verify=False, timeout=10)
                
                if res_lineup.status_code == 200:
                    lineup_json = res_lineup.json()
                    for side in ['home', 'away']:
                        for p in lineup_json.get(side, {}).get('players', []):
                            stats = p.get('statistics', {})
                            if stats:
                                p_obj = p.get('player', {})
                                p_id_raw = p_obj.get('id')
                                
                                if p_id_raw:
                                    p_id_str = str(p_id_raw)
                                    c = coords.get(p_id_str, {'x': None, 'y': None})                     
                                    
                                    avg_x = c.get('x')
                                    avg_y = c.get('y')
                                    
                                    if avg_x is not None:
                                        sigla = identificar_posicao_opta(avg_x, avg_y)
                                    else:
                                        sigla = "N/A"

                                    all_players_data.append({
                                        "Temporada": ano,
                                        "Rodada": round_num,
                                        "Jogador": p_obj.get('name'),
                                        "Time": match[f'{side}Team']['name'],
                                        "Avg_X": avg_x,
                                        "Avg_Y": avg_y,
                                        "Sigla_Opta": sigla,
                                        "Posicao_Real": mapa_tatico.get(sigla, "Outro"),
                                        **stats
                                    })
                time.sleep(0.1) 
        except Exception:
            continue

    if all_players_data:
        df = pd.DataFrame(all_players_data)
        filename = f"brasileirao_{ano}_opta_final.csv"
        cols = ["Temporada", "Rodada", "Jogador", "Time", "Avg_X", "Avg_Y", "Sigla_Opta", "Posicao_Real"]
        df = df[cols + [c for c in df.columns if c not in cols]]
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nSucesso: {filename} gerado com {len(df)} linhas.")

print("\nExtração finalizada com sucesso!")


--- Extraindo Temporada 2025 (ID Sofa: 72034) ---
Processando Rodada 38/38
Sucesso: brasileirao_2025_opta_final.csv gerado com 17396 linhas.

--- Extraindo Temporada 2024 (ID Sofa: 58766) ---
